<a href="https://colab.research.google.com/github/Hussam3d/Hussam3d/blob/main/best_of_5_Axis_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
# @title 🚀 5-AXIS ULTIMATE PhD DESIGNER (Helix + Mesh + Textures + Bend Control) { display-mode: "form" }
# @markdown ---
# @markdown ### 🛠️ 1. GLOBAL ACTIONS
preview_style = 'line' # @param ["line", "tube"]
download_gcode = False # @param {type:"boolean"}
design_name = '5axis_phd_master' # @param {type:"string"}

# @markdown ---
# @markdown ### 🌀 2. SMART HELIX SETTINGS
helix_radius_percent = 25 # @param {type:"slider", min:0, max:50, step:1}
helix_revolutions = 2 # @param {type:"slider", min:0, max:10, step:0.1}

# @markdown ---
# @markdown ### 🕸️ 3. POROUS MESH (WIREFRAME)
enable_wireframe_mesh = False # @param {type:"boolean"}
mesh_density = 8 # @param {type:"slider", min:4, max:32, step:1}
# @markdown > *Note: This drops flow to 0% at intervals to create a structural net/lattice.*

# @markdown ---
# @markdown ### 💠 4. SHAPE MORPHING (LOFT)
bottom_shape = 11 # @param {type:"slider", min:3, max:11, step:1}
bottom_size = 20 # @param {type:"slider", min:5, max:100, step:0.5}
enable_middle_shape = True # @param {type:"boolean"}
middle_shape = 11 # @param {type:"slider", min:3, max:11, step:1}
middle_size = 20 # @param {type:"slider", min:5, max:100, step:0.5}
top_shape = 3 # @param {type:"slider", min:3, max:11, step:1}
top_size = 20 # @param {type:"slider", min:5, max:100, step:0.5}

# @markdown ---
# @markdown ### 🎢 5. PATH & KINEMATICS
total_length = 80 # @param {type:"slider", min:20, max:300, step:1}
bend_angle = 0 # @param {"type":"slider","min":0,"max":180,"step":5}
bend_location = 50 # @param {type:"slider", min:0, max:100, step:1}
twist_angle = 0 # @param {"type":"slider","min":-360,"max":360,"step":5}
enable_wedge_layer = True # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🎨 6. THE LIVING SURFACE (TEXTURES)
texture_style = 'Knurled' # @param ['Smooth', 'Ribbed', 'Knurled', 'Bamboo', 'Fluted', 'Dimpled', 'Spiked']
texture_depth = 1.0 # @param {type:"slider", min:0.0, max:5.0, step:0.1}
texture_density = 12 # @param {type:"slider", min:4, max:32, step:1}
enable_texture_fading = False # @param {type:"boolean"}

# @markdown ---
# @markdown ### ⚙️ 7. PRINTER SETTINGS
EW = 0.4 # @param {type:"slider", min:0.1, max:1.2, step:0.01}
EH = 0.22 # @param {type:"slider", min:0.05, max:0.6, step:0.01}
segs_per_layer = 180 # @param {type:"slider", min:64, max:360, step:1}
tilt_axis_letter = "B" # @param {type:"string"}
invert_tilt_direction = False # @param {type:"boolean"}

# --- AUTO-INSTALLER ---
import subprocess, sys
try:
    import fullcontrol as fc
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fullcontrol", "-q"])
    import fullcontrol as fc
import math
from google.colab import files

def parse_shape(val): return 64 if val == 11 else val
def get_polygon_radius(theta, edges, size):
    radius = size / 2.0
    if edges >= 50: return radius
    slice_angle = 2 * math.pi / edges
    local_angle = (theta % slice_angle) - (slice_angle / 2)
    return radius / math.cos(local_angle)

# --- Geometry Setup ---
max_allowed_radius = (bottom_size / 2.0) * (helix_radius_percent / 100.0)
initial_z = EH * 0.4
bend_angle_rad = math.radians(bend_angle)
total_twist_rad = math.radians(twist_angle)
max_r = max([bottom_size, middle_size, top_size])
safe_bend_radius = max_r * 1.5
arc_L = bend_angle_rad * safe_bend_radius
straight_total = max(0.1, total_length - arc_L)
L1_dist = straight_total * (bend_location / 100.0)
L2_dist = L1_dist + arc_L
total_steps = int(total_length / (EH / segs_per_layer))

steps = []; tilt_angles = []; flow_percentages = []
steps.append(fc.Extruder(on=False))
steps.append(fc.Point(x=max_allowed_radius + (bottom_size/2), y=0, z=initial_z))
steps.append(fc.Extruder(on=True))

for step in range(total_steps):
    L = step * (EH / segs_per_layer); theta = step * (math.pi * 2 / segs_per_layer)
    u = min(1.0, L / total_length)

    # Helix
    hx_theta = u * (2 * math.pi * helix_revolutions)
    cx_h, cy_h = max_allowed_radius * math.cos(hx_theta), max_allowed_radius * math.sin(hx_theta)

    # Morphing
    shps = [parse_shape(bottom_shape), parse_shape(middle_shape if enable_middle_shape else top_shape), parse_shape(top_shape)]
    szs = [bottom_size, middle_size if enable_middle_shape else top_size, top_size]
    idx = int(u * (len(shps)-1))
    t = (u * (len(shps)-1)) - idx
    if idx >= len(shps)-1: idx = len(shps)-2; t = 1.0
    sm_t = (1.0 - math.cos(t * math.pi)) / 2.0
    r_curr = get_polygon_radius(theta, shps[idx], szs[idx]) * (1-sm_t) + get_polygon_radius(theta, shps[idx+1], szs[idx+1]) * sm_t

    # Texture
    depth = texture_depth * (math.sin(u * math.pi) if enable_texture_fading else 1.0)
    v_fr = texture_density * (L / (max_r/2)) * 2
    if texture_style == 'Ribbed': r_curr += depth * math.sin(texture_density * theta)
    elif texture_style == 'Knurled': r_curr += depth * math.sin(texture_density * theta) * math.cos(v_fr)
    elif texture_style == 'Bamboo': r_curr += depth * math.sin(v_fr)
    elif texture_style == 'Fluted': r_curr -= depth * (math.sin(texture_density * theta)**4)
    elif texture_style == 'Dimpled': r_curr -= depth * abs(math.sin(texture_density * theta) * math.cos(v_fr))
    elif texture_style == 'Spiked': r_curr += depth * (max(0, math.sin(texture_density * theta) * math.cos(v_fr))**4)

    # Twist & Coordinates
    tw = total_twist_rad * ((1.0 - math.cos(u * math.pi)) / 2.0)
    x_l, y_l = r_curr * math.cos(theta + tw), r_curr * math.sin(theta + tw)

    # Path & Wedge
    flow = 100
    if L <= L1_dist: cx, cz, alpha = 0, initial_z + L, 0
    elif L <= L2_dist:
        a_curr = (L - L1_dist) / safe_bend_radius; alpha = a_curr
        cx, cz = safe_bend_radius * (1 - math.cos(alpha)), initial_z + L1_dist + safe_bend_radius * math.sin(alpha)
        if enable_wedge_layer: flow = max(10, min(300, int(round(100 * (1.0 - (x_l / (max_r*1.5)))))))
    else:
        dist = L - L2_dist; alpha = bend_angle_rad
        cx = safe_bend_radius * (1 - math.cos(alpha)) + dist * math.sin(alpha)
        cz = initial_z + L1_dist + safe_bend_radius * math.sin(alpha) + dist * math.cos(alpha)

    # MESH LOGIC (E-Axis Interrupts)
    if enable_wireframe_mesh:
        # Intersecting Sine Waves create the holes
        mesh_v = math.sin(mesh_density * theta) * math.cos(mesh_density * L / (max_r/2))
        if mesh_v > 0.4: flow = 0 # Drop flow to zero for the gaps

    x = cx_h + cx + x_l * math.cos(alpha)
    y = cy_h + y_l
    z = cz - x_l * math.sin(alpha)

    steps.append(fc.Point(x=x, y=y, z=z))
    tilt_angles.append(math.degrees(alpha) * (-1 if invert_tilt_direction else 1))
    flow_percentages.append(flow)

fc.transform(steps, 'plot', fc.PlotControls(style=preview_style, zoom=0.7))

if download_gcode:
    gc = fc.transform(steps, 'gcode', fc.GcodeControls(printer_name='prusa_i3', initialization_data={'print_speed': 1000, 'nozzle_temp': 210, 'bed_temp': 55, 'extrusion_width': EW, 'extrusion_height': EH}))
    new_gc = []; t_idx = 0; c_flow = 100
    for line in gc.split('\n'):
        if line.startswith('G1') and 'X' in line:
            if t_idx < len(tilt_angles):
                if flow_percentages[t_idx] != c_flow:
                    new_gc.append(f"M221 S{flow_percentages[t_idx]}"); c_flow = flow_percentages[t_idx]
                line += f" {tilt_axis_letter}{tilt_angles[t_idx]:.2f}"; t_idx += 1
        new_gc.append(line)
    with open(f'{design_name}.gcode', 'w') as f: f.write('\n'.join(new_gc))
    files.download(f'{design_name}.gcode')